In [2]:
%pip install langchain-huggingface dotenv scikit-learn requests sentence-transformers

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached tokenizers-0.22.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 18.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 16.5 MB/s eta 0:00:00
Using cached

In [1]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import requests
import os
from dotenv import load_dotenv
import re

load_dotenv()
ED_API_KEY = os.getenv("ED_API_KEY")

/Users/oscar/Developer/edstem-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [4]:
def fetch_threads(limit=50, sort="new"):
  """
  Fetches the Ed Thread API, guaraanteeing <limit> threads are returned. The 
  Ed API sets a hard limit of 100, so if more are needed, multiple requests
  are made with different offsets.
  TODO: handle rate limiting, fetch all threads if limit is None
  """
  threads = []
  while len(threads) < limit:
    res = requests.get(
      url=f"https://us.edstem.org/api/courses/74827/threads?limit={min(100, limit - len(threads))}&offset={len(threads)}&sort={sort}",
      headers={"Authorization": f"Bearer {ED_API_KEY}"}
    )
    data = res.json()
    threads.extend(data['threads'])
    if len(data['threads']) < 100:
      break
  threads = {
    thread["id"]: 
      {
        "title": thread["title"],
        "content": clean_xml_tags(thread["content"]).strip(),
        "title_embedding": embeddings.embed_query(thread["title"]),
        "content_embedding": embeddings.embed_query(clean_xml_tags(thread["content"]).strip())
  } for thread in threads}
  return threads

def getCommentsandAnswers(thread_ids):
    """
    Fetch all comments (including those on answers) for a list of thread IDs.
    Returns a list of comment texts.
    """
    def collect_all_comments(comment_list):
      """Recursively collect all comment texts from a list of comments."""
      texts = []
      for comment in comment_list:
          texts.append(comment.get('document', comment.get('content', '')))
          # Recursively collect nested comments
          if comment.get('comments'):
              texts.extend(collect_all_comments(comment['comments']))
      return texts
    

    all_comments = []
    all_answers = []
    # Goes through each thread id and gets the specific comments that it is
    # trying to get.
    for tid in thread_ids:
        res = requests.get(
            url=f"https://us.edstem.org/api/threads/{tid}",
            headers={"Authorization": f"Bearer {ED_API_KEY}"},
        )
        if res.status_code != 200:
            print(f"Failed to fetch thread {tid}: {res.status_code}")
            continue
        
        thread = res.json().get('thread', {})

        all_comments.extend(collect_all_comments(thread.get('comments', [])))
            
        for answer in thread.get('answers', []):
            all_answers.append(answer.get('document', answer.get('content', '')))
            all_comments.extend(collect_all_comments(answer.get('comments', [])))
    return all_comments, all_answers

def pre_processing(text):
  """Lowercase, remove punctuation, and extra whitespace."""
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  text = re.sub(r'\s+', ' ', text)     
  return text.strip()


def search_threads(query, limit=20, sort="relevance", category=None, from_date=None, to_date=None):
  """
  Whole point of this function is to get the embeddings and the text of each 
  thread that matches the search query.

  """
  base = {
    "query": query,
    "limit": limit,
    "sort": sort,
    "category": category,
    "from_date": from_date,
    "to_date": to_date,
  }
  params = {k: v for k, v in base.items() if v is not None}
  res = requests.get(
    url="https://us.edstem.org/api/courses/74827/threads/search",
    headers={"Authorization": f"Bearer {ED_API_KEY}"},
    params=params
  )
  threads = res.json()['threads']
  result = {}
    
  for thread in threads:
    comments, answers = getCommentsandAnswers([thread["id"]])
    comment_texts = []
    answer_texts = []

    for comment in comments:
        comment_texts.append(clean_xml_tags(comment).strip())
    for answer in answers:
        answer_texts.append(clean_xml_tags(answer).strip())


    result[thread["id"]] = {
      "title": thread["title"],
      "content": pre_processing(clean_xml_tags(thread["title"]) + " " + 
                                clean_xml_tags(thread["content"]).strip() + " " + 
      " ".join(answer_texts) + " " + " ".join(comment_texts)),
    }

  for thread_id, data in result.items():
    # Generate title and content embeddings
    data["content_embedding"] = embeddings.embed_query(data["content"])

  return result

def clean_xml_tags(text):
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)


In [5]:
text = search_threads('diffusion math',
                limit=5,
                sort='relevance')

print(text)

{6436163: {'title': 'posterior and prior', 'content': 'posterior and prior just to clarify in elbo for diffusion and vae what is the posterior and what is the prior for vaes the prior is the distribution of our latent space which we represent with pz the assumption is that this is normally distributed the posterior is the distribution that our encoder approximates qz x in general the prior refers to the probability of your hypothesis the latent space value z before seeing any data the posterior is the revised probability after observing data the data point x for diffusion models it gets more complicated and im not confident enough to answer the idea is similar but the denoising process and time sampling make it harder i think posterior and prior are more general terms looking at this try to understand which part of diffusion is the prior and which is the posterior', 'content_embedding': [0.013128656893968582, -0.019864941015839577, 0.011099680326879025, -0.041104756295681, 0.0112218512

In [6]:
def find_top_k_results(q, threads, k=5):
  """
  Find the top k results from thread content based on cosine similarity to query q.
  Returns a list of (thread_id, similarity) tuples.
  """

  results = sorted(((item[0], cosine_similarity([q], [item[1]['content_embedding']])[0][0]) for item in threads.items()), key=lambda item: item[1], reverse=True)
  for thread_id, sim in results[:k]:
    print(f"Thread ID: {thread_id}, Similarity: {sim:.4f}")
    print(f"Title: {threads[thread_id]['title']}")
    print(f"Content: {threads[thread_id]['content'][:200]}...")
    print()
  return results[:k]

In [7]:
all_threads = fetch_threads(limit=600, sort="new")
len(all_threads)

583

In [8]:
q = "Can someone explain transformers"
results = find_top_k_results(embeddings.embed_query(q), all_threads, k=5)

Thread ID: 6332379, Similarity: 0.4605
Title: Why are Transformers so much faster than e.g. LSTMs?
Content: One thing some students still seem confused about is why transformers are so much more popular, if they are still autoregressive (i.e. they are generating token after token left to right). The answer ...

Thread ID: 6436211, Similarity: 0.4206
Title: question abt CLIP/swin transformers
Content: are CLIP and Swin transformers considered self-supervised or supervised learning?...

Thread ID: 6332365, Similarity: 0.3684
Title: Transformer Accuracy Right, but number of layers not
Content: My autograder says the number of layers is not right for the transformer architecture but somehow the accuracy is, how can this be ? Is this just pure luck...

Thread ID: 6432462, Similarity: 0.3252
Title: Practice midterm q2 8)
Content: Transforms samples from a unit normal distribution to samples from the data distribution.Does this apply to all transformers not just the image ones?...

Thread ID:

In [9]:
def compare_search_results(query, vector_results, native_results, k=5):
    """
    Compare vector search results with native EdStem search results.
    
    Args:
        query: The search query
        vector_results: List of (thread_id, similarity_score) tuples from vector search
        native_results: Dict of thread_id -> thread_data from native search
        k: Number of top results to compare
    
    Returns:
        Dict with various comparison metrics
    """
    # Extract thread IDs
    vector_ids = [tid for tid, _ in vector_results[:k]]
    native_ids = list(native_results.keys())[:k]
    
    # Calculate overlap
    overlap_ids = set(vector_ids) & set(native_ids)
    overlap_percentage = len(overlap_ids) / k * 100 if k > 0 else 0
    
    # Find position of vector results in native results
    position_analysis = {}
    for tid, score in vector_results[:k]:
        if tid in native_ids:
            position_analysis[tid] = {
                'vector_position': vector_ids.index(tid) + 1,
                'native_position': native_ids.index(tid) + 1,
                'vector_score': score,
                'position_diff': (vector_ids.index(tid) + 1) - (native_ids.index(tid) + 1)
            }
    
    # Calculate average position improvement (negative = better in vector, positive = better in native)
    avg_position_diff = sum([p['position_diff'] for p in position_analysis.values()]) / len(position_analysis) if position_analysis else 0
    
    # Find results unique to each method
    vector_only = set(vector_ids) - set(native_ids)
    native_only = set(native_ids) - set(vector_ids)
    
    # Score distribution
    vector_scores = [score for _, score in vector_results[:k]]
    avg_vector_score = sum(vector_scores) / len(vector_scores) if vector_scores else 0
    
    return {
        'query': query,
        'overlap_percentage': overlap_percentage,
        'overlap_count': len(overlap_ids),
        'overlap_ids': list(overlap_ids),
        'position_analysis': position_analysis,
        'avg_position_diff': avg_position_diff,
        'vector_only': list(vector_only),
        'native_only': list(native_only),
        'vector_scores': vector_scores,
        'avg_vector_score': avg_vector_score,
        'vector_results': vector_results[:k],
        'native_results': native_ids
    }


In [10]:
def evaluate_query(query, all_threads, k=5):
    """
    Evaluate a single query by comparing vector search vs native search.
    
    Returns:
        Comparison metrics dict
    """
    # Vector search: find top k using embeddings
    query_embedding = embeddings.embed_query(query)
    vector_results = sorted(
        ((tid, cosine_similarity([query_embedding], [data['content_embedding']])[0][0]) 
         for tid, data in all_threads.items()),
        key=lambda x: x[1],
        reverse=True
    )[:k]
    
    # Native search: use EdStem API
    native_results = search_threads(query, limit=k, sort='relevance')
    
    # Compare
    comparison = compare_search_results(query, vector_results, native_results, k)
    
    return comparison


In [11]:
def print_comparison(comparison):
    """Pretty print comparison results."""
    print(f"\n{'='*60}")
    print(f"Query: '{comparison['query']}'")
    print(f"{'='*60}\n")
    
    print(f"📊 OVERLAP METRICS:")
    print(f"  Overlap: {comparison['overlap_count']}/{len(comparison['vector_results'])} ({comparison['overlap_percentage']:.1f}%)")
    print(f"  Vector-only results: {len(comparison['vector_only'])}")
    print(f"  Native-only results: {len(comparison['native_only'])}")
    
    print(f"\n📈 POSITION ANALYSIS:")
    if comparison['position_analysis']:
        print(f"  Average position difference: {comparison['avg_position_diff']:.2f}")
        print(f"  (Negative = better in vector search, Positive = better in native)")
        print(f"\n  Detailed positions:")
        for tid, info in comparison['position_analysis'].items():
            print(f"    Thread {tid}:")
            print(f"      Vector: #{info['vector_position']} (score: {info['vector_score']:.4f})")
            print(f"      Native: #{info['native_position']}")
            print(f"      Difference: {info['position_diff']:+d} positions")
    else:
        print("  No overlapping results to compare positions")
    
    print(f"\n🎯 SCORE DISTRIBUTION:")
    print(f"  Average vector similarity score: {comparison['avg_vector_score']:.4f}")
    print(f"  Score range: {min(comparison['vector_scores']):.4f} - {max(comparison['vector_scores']):.4f}")
    
    print(f"\n🔍 VECTOR-ONLY RESULTS (not in native top {len(comparison['vector_results'])}):")
    for tid in comparison['vector_only']:
        score = next((s for t, s in comparison['vector_results'] if t == tid), None)
        print(f"  Thread {tid}: score {score:.4f}")
    
    print(f"\n🔍 NATIVE-ONLY RESULTS (not in vector top {len(comparison['vector_results'])}):")
    for tid in comparison['native_only']:
        print(f"  Thread {tid}")
    
    print(f"\n{'='*60}\n")


In [12]:
# Example: Compare results for a query
query = "Can someone explain transformers"
comparison = evaluate_query(query, all_threads, k=5)
print_comparison(comparison)



Query: 'Can someone explain transformers'

📊 OVERLAP METRICS:
  Overlap: 3/5 (60.0%)
  Vector-only results: 2
  Native-only results: 2

📈 POSITION ANALYSIS:
  Average position difference: 0.33
  (Negative = better in vector search, Positive = better in native)

  Detailed positions:
    Thread 6332379:
      Vector: #1 (score: 0.4605)
      Native: #3
      Difference: -2 positions
    Thread 6436211:
      Vector: #2 (score: 0.4206)
      Native: #1
      Difference: +1 positions
    Thread 6432462:
      Vector: #4 (score: 0.3252)
      Native: #2
      Difference: +2 positions

🎯 SCORE DISTRIBUTION:
  Average vector similarity score: 0.3699
  Score range: 0.2749 - 0.4605

🔍 VECTOR-ONLY RESULTS (not in native top 5):
  Thread 6272505: score 0.2749
  Thread 6332365: score 0.3684

🔍 NATIVE-ONLY RESULTS (not in vector top 5):
  Thread 6431379
  Thread 6577439




In [15]:
def batch_evaluate(queries, all_threads, k=5):
    """
    Evaluate multiple queries and aggregate metrics.
    
    Returns:
        Dict with aggregated metrics across all queries
    """
    results = []
    for query in queries:
        try:
            comparison = evaluate_query(query, all_threads, k)
            results.append(comparison)
        except Exception as e:
            print(f"Error evaluating query '{query}': {e}")
            continue
    
    if not results:
        return None
    
    # Aggregate metrics
    avg_overlap = sum(r['overlap_percentage'] for r in results) / len(results)
    avg_position_diff = sum(r['avg_position_diff'] for r in results if r['position_analysis']) / len([r for r in results if r['position_analysis']]) if any(r['position_analysis'] for r in results) else 0
    avg_vector_score = sum(r['avg_vector_score'] for r in results) / len(results)
    
    total_overlap_count = sum(r['overlap_count'] for r in results)
    total_queries = len(results)
    total_possible = total_queries * k
    
    # Win rate: % of queries where vector search finds results not in native top k
    queries_with_vector_only = sum(1 for r in results if len(r['vector_only']) > 0)
    win_rate = queries_with_vector_only / total_queries * 100 if total_queries > 0 else 0
    
    # Coverage: % of queries that return vector results
    coverage = 100.0  # Always 100% if we have embeddings
    
    return {
        'total_queries': total_queries,
        'avg_overlap_percentage': avg_overlap,
        'total_overlap_count': total_overlap_count,
        'total_possible_overlap': total_possible,
        'overlap_rate': total_overlap_count / total_possible * 100 if total_possible > 0 else 0,
        'avg_position_diff': avg_position_diff,
        'avg_vector_score': avg_vector_score,
        'win_rate': win_rate,
        'coverage': coverage,
        'individual_results': results
    }


In [16]:
def print_aggregate_metrics(aggregate):
    """Print aggregated metrics across multiple queries."""
    print(f"\n{'='*70}")
    print(f"AGGREGATE METRICS ({aggregate['total_queries']} queries)")
    print(f"{'='*70}\n")
    
    print(f"📊 OVERLAP METRICS:")
    print(f"  Average overlap: {aggregate['avg_overlap_percentage']:.1f}%")
    print(f"  Total overlap: {aggregate['total_overlap_count']}/{aggregate['total_possible_overlap']} ({aggregate['overlap_rate']:.1f}%)")
    
    print(f"\n📈 POSITION METRICS:")
    print(f"  Average position difference: {aggregate['avg_position_diff']:.2f}")
    print(f"  (Negative = vector search ranks higher, Positive = native ranks higher)")
    
    print(f"\n🎯 SCORE METRICS:")
    print(f"  Average vector similarity score: {aggregate['avg_vector_score']:.4f}")
    
    print(f"\n🏆 WIN RATE:")
    print(f"  Queries where vector search found unique results: {aggregate['win_rate']:.1f}%")
    
    print(f"\n📈 COVERAGE:")
    print(f"  Query coverage: {aggregate['coverage']:.1f}%")
    
    print(f"\n{'='*70}\n")


In [17]:
# Test with multiple queries
test_queries = [
    "Can someone explain transformers",
    "diffusion math",
    "VAE encoder decoder",
    "backpropagation",
    "attention mechanism"
]

aggregate = batch_evaluate(test_queries, all_threads, k=5)
if aggregate:
    print_aggregate_metrics(aggregate)
    
    # Show individual results
    print("\n" + "="*70)
    print("INDIVIDUAL QUERY RESULTS")
    print("="*70)
    for result in aggregate['individual_results']:
        print_comparison(result)



AGGREGATE METRICS (5 queries)

📊 OVERLAP METRICS:
  Average overlap: 36.0%
  Total overlap: 9/25 (36.0%)

📈 POSITION METRICS:
  Average position difference: 0.02
  (Negative = vector search ranks higher, Positive = native ranks higher)

🎯 SCORE METRICS:
  Average vector similarity score: 0.4543

🏆 WIN RATE:
  Queries where vector search found unique results: 100.0%

📈 COVERAGE:
  Query coverage: 100.0%



INDIVIDUAL QUERY RESULTS

Query: 'Can someone explain transformers'

📊 OVERLAP METRICS:
  Overlap: 3/5 (60.0%)
  Vector-only results: 2
  Native-only results: 2

📈 POSITION ANALYSIS:
  Average position difference: 0.33
  (Negative = better in vector search, Positive = better in native)

  Detailed positions:
    Thread 6332379:
      Vector: #1 (score: 0.4605)
      Native: #3
      Difference: -2 positions
    Thread 6436211:
      Vector: #2 (score: 0.4206)
      Native: #1
      Difference: +1 positions
    Thread 6432462:
      Vector: #4 (score: 0.3252)
      Native: #2
      Di

In [18]:
# Additional analysis: Score threshold analysis
def analyze_score_thresholds(comparison, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8]):
    """Analyze how many results would be shown at different score thresholds."""
    print(f"\n{'='*60}")
    print(f"SCORE THRESHOLD ANALYSIS for: '{comparison['query']}'")
    print(f"{'='*60}\n")
    
    print(f"{'Threshold':<12} {'Results':<10} {'Avg Score':<12} {'Overlap':<10}")
    print("-" * 60)
    
    for threshold in thresholds:
        filtered = [(tid, score) for tid, score in comparison['vector_results'] if score >= threshold]
        if filtered:
            filtered_ids = [tid for tid, _ in filtered]
            overlap = len(set(filtered_ids) & set(comparison['native_results']))
            avg_score = sum(score for _, score in filtered) / len(filtered)
            print(f"{threshold:<12.2f} {len(filtered):<10} {avg_score:<12.4f} {overlap}/{len(filtered):<10}")
        else:
            print(f"{threshold:<12.2f} {'0':<10} {'N/A':<12} {'0':<10}")
    
    print(f"\n{'='*60}\n")

# Run threshold analysis on a query
query = "Can someone explain transformers"
comparison = evaluate_query(query, all_threads, k=10)  # Get more results for threshold analysis
analyze_score_thresholds(comparison)



SCORE THRESHOLD ANALYSIS for: 'Can someone explain transformers'

Threshold    Results    Avg Score    Overlap   
------------------------------------------------------------
0.30         4          0.3937       3/4         
0.40         2          0.4405       2/2         
0.50         0          N/A          0         
0.60         0          N/A          0         
0.70         0          N/A          0         
0.80         0          N/A          0         




In [19]:
import json
from datetime import datetime

def export_evaluation_results(aggregate, filename=None):
    """Export evaluation results to JSON for further analysis."""
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"evaluation_results_{timestamp}.json"
    
    # Prepare export data (remove embeddings to keep file size manageable)
    export_data = {
        'timestamp': datetime.now().isoformat(),
        'total_queries': aggregate['total_queries'],
        'metrics': {
            'avg_overlap_percentage': aggregate['avg_overlap_percentage'],
            'total_overlap_count': aggregate['total_overlap_count'],
            'total_possible_overlap': aggregate['total_possible_overlap'],
            'overlap_rate': aggregate['overlap_rate'],
            'avg_position_diff': aggregate['avg_position_diff'],
            'avg_vector_score': aggregate['avg_vector_score'],
            'win_rate': aggregate['win_rate'],
            'coverage': aggregate['coverage']
        },
        'individual_results': []
    }
    
    # Add individual results (without full embeddings)
    for result in aggregate['individual_results']:
        export_data['individual_results'].append({
            'query': result['query'],
            'overlap_percentage': result['overlap_percentage'],
            'overlap_count': result['overlap_count'],
            'avg_position_diff': result['avg_position_diff'],
            'avg_vector_score': result['avg_vector_score'],
            'vector_only_count': len(result['vector_only']),
            'native_only_count': len(result['native_only']),
            'vector_results': [(tid, round(score, 4)) for tid, score in result['vector_results']],
            'native_results': result['native_results'],
            'position_analysis': {
                tid: {
                    'vector_position': info['vector_position'],
                    'native_position': info['native_position'],
                    'vector_score': round(info['vector_score'], 4),
                    'position_diff': info['position_diff']
                }
                for tid, info in result['position_analysis'].items()
            }
        })
    
    with open(filename, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"✅ Results exported to {filename}")
    return filename

# Export results if aggregate exists
if 'aggregate' in locals() and aggregate:
    export_evaluation_results(aggregate)


✅ Results exported to evaluation_results_20251214_133556.json


In [20]:
# Optional: Create a summary report
def create_summary_report(aggregate):
    """Create a text summary report of the evaluation."""
    report = []
    report.append("="*70)
    report.append("VECTOR SEARCH vs NATIVE SEARCH - EVALUATION REPORT")
    report.append("="*70)
    report.append(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"\nTotal Queries Evaluated: {aggregate['total_queries']}")
    report.append("\n" + "-"*70)
    
    report.append("\n📊 KEY METRICS:")
    report.append(f"  • Average Overlap: {aggregate['avg_overlap_percentage']:.1f}%")
    report.append(f"  • Overall Overlap Rate: {aggregate['overlap_rate']:.1f}%")
    report.append(f"  • Average Position Difference: {aggregate['avg_position_diff']:.2f}")
    report.append(f"  • Win Rate (unique vector results): {aggregate['win_rate']:.1f}%")
    report.append(f"  • Average Vector Score: {aggregate['avg_vector_score']:.4f}")
    
    report.append("\n" + "-"*70)
    report.append("\n📈 INTERPRETATION:")
    if aggregate['avg_position_diff'] < 0:
        report.append("  ✓ Vector search tends to rank relevant results higher than native search")
    elif aggregate['avg_position_diff'] > 0:
        report.append("  ⚠ Native search tends to rank relevant results higher than vector search")
    else:
        report.append("  = Both methods rank overlapping results similarly")
    
    if aggregate['win_rate'] > 50:
        report.append(f"  ✓ Vector search finds unique results in {aggregate['win_rate']:.1f}% of queries")
    else:
        report.append(f"  ⚠ Vector search finds unique results in only {aggregate['win_rate']:.1f}% of queries")
    
    if aggregate['avg_overlap_percentage'] > 60:
        report.append(f"  ✓ High overlap ({aggregate['avg_overlap_percentage']:.1f}%) suggests good alignment")
    elif aggregate['avg_overlap_percentage'] < 40:
        report.append(f"  ⚠ Low overlap ({aggregate['avg_overlap_percentage']:.1f}%) suggests different coverage")
    else:
        report.append(f"  = Moderate overlap ({aggregate['avg_overlap_percentage']:.1f}%) - complementary results")
    
    report.append("\n" + "="*70)
    
    report_text = "\n".join(report)
    print(report_text)
    
    # Save to file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"evaluation_report_{timestamp}.txt"
    with open(filename, 'w') as f:
        f.write(report_text)
    print(f"\n✅ Report saved to {filename}")
    
    return report_text

# Generate summary report
if 'aggregate' in locals() and aggregate:
    create_summary_report(aggregate)


VECTOR SEARCH vs NATIVE SEARCH - EVALUATION REPORT

Generated: 2025-12-14 13:35:58

Total Queries Evaluated: 5

----------------------------------------------------------------------

📊 KEY METRICS:
  • Average Overlap: 36.0%
  • Overall Overlap Rate: 36.0%
  • Average Position Difference: 0.02
  • Win Rate (unique vector results): 100.0%
  • Average Vector Score: 0.4543

----------------------------------------------------------------------

📈 INTERPRETATION:
  ⚠ Native search tends to rank relevant results higher than vector search
  ✓ Vector search finds unique results in 100.0% of queries
  ⚠ Low overlap (36.0%) suggests different coverage


✅ Report saved to evaluation_report_20251214_133558.txt


In [153]:
search = search_threads(q, limit=5)
search_sim = {thread_id: cosine_similarity([embeddings.embed_query(q)], [data['content_embedding']])[0][0] for thread_id, data in search.items()}
for thread_id, sim in search_sim.items():
  print(f"Thread ID: {thread_id}, Similarity: {sim:.4f}")
  print(f"Title: {search[thread_id]['title']}")
  print(f"Content: {search[thread_id]['content'][:200]}...")
  print()

Thread ID: 6436211, Similarity: 0.4238
Title: question abt CLIP/swin transformers
Content: question abt clipswin transformers are clip and swin transformers considered selfsupervised or supervised learning clip isnt a transformer in and of itself but it uses encoders which are often transfo...

Thread ID: 6432462, Similarity: 0.4021
Title: Practice midterm q2 8)
Content: practice midterm q2 8 transforms samples from a unit normal distribution to samples from the data distributiondoes this apply to all transformers not just the image ones no since nlp transformers for ...

Thread ID: 6332379, Similarity: 0.4164
Title: Why are Transformers so much faster than e.g. LSTMs?
Content: why are transformers so much faster than eg lstms one thing some students still seem confused about is why transformers are so much more popular if they are still autoregressive ie they are generating...

Thread ID: 6577439, Similarity: 0.1639
Title: ALiBi (Attention with Linear Biases)
Content: alibi attention 